# Analyse de manuscrits médiévaux avec YOLO-gen

**Référence :** Torres Aguilar, S. (2025). *From Codicology to Code: A Comparative Study of Transformer and YOLO-based Detectors for Layout Analysis in Historical Documents*. arXiv:2506.20326  
**Modèle :** `magistermilitum/YOLO_manuscripts` (Hugging Face, licence MIT)

## 1. Installation des dépendances

In [52]:
#!pip install ultralytics huggingface_hub pillow matplotlib ipykernel

## 2. Imports et constantes

In [53]:
import os
import sys
import json
from pathlib import Path
from collections import defaultdict

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon, Patch
from PIL import Image
from huggingface_hub import hf_hub_download
from ultralytics import YOLO

#IMAGE_PATH  = "Le_Roman_de_la_Rose_[...]Guillaume_de_btv1b6000369q_8.jpeg"
IMAGE_PATH  = r"C:\Users\djame\Documents\GitHub\Analyse-de-manuscrits\Exercice 3\Le_Roman_de_la_Rose_[...]Guillaume_de_btv1b6000369q_8 - Copie.jpeg"
#IMAGE_PATH = r"C:\Users\djame\Documents\GitHub\Analyse-de-manuscrits\Exercice 3\not_detected_croped.jpeg"
OUTPUT_DIR  = Path("yologen_output")
HF_MODEL_ID = "magistermilitum/YOLO_manuscripts"
HF_TOKEN    = None   # ou "hf_..."
CONF_FINAL  = 0.10

## 3. Fonction `load_model`

Télécharge `best.pt` depuis Hugging Face et retourne un objet `YOLO` prêt à l'inférence.

In [54]:
def load_model() -> YOLO:
    # Résolution du token : variable globale → variable d'environnement → cache HF
    token = HF_TOKEN or os.environ.get("HF_TOKEN") or None

    try:
        model_path = hf_hub_download(
            repo_id=HF_MODEL_ID,
            filename="best.pt",
            token=token
        )
    except Exception as e:
        if "403" in str(e) or "401" in str(e):
            sys.exit(
                "[ERREUR] Accès refusé (403/401). "
                "Définissez HF_TOKEN avec un token Hugging Face valide "
                "ou lancez `huggingface-cli login`."
            )
        raise

    model = YOLO(model_path)
    print(f"Modèle chargé depuis : {model_path}")
    print(f"Classes détectables  : {model.names}")
    return model

## 4. Fonction `diagnose_thresholds`

Teste plusieurs seuils décroissants et retourne les résultats au premier seuil ayant produit au moins une détection.

In [ ]:
def diagnose_thresholds(model: YOLO, image_path: str):
    thresholds = [0.50, 0.25, 0.10, 0.05, 0.01]

    print(f"{'Seuil':>8} | {'Détections':>10}")
    print("-" * 22)

    best_results = None
    best_conf    = None

    for conf in thresholds:
        results = model.predict(
            source=image_path,
            conf=conf,
            iou=0.45,
            imgsz=1080,
            verbose=False
        )

        n = 0
        for r in results:
            if r.obb is not None:
                n += len(r.obb)

        print(f"{conf:>8.2f} | {n:>10}")

        if n > 0 and conf <= CONF_FINAL and best_results is None:
            best_results = results
            best_conf    = conf

    if best_results is None:
        best_results = results
        best_conf    = thresholds[-1]

    print(f"\nSeuil retenu : {best_conf}")
    return best_results, best_conf

## 5. Fonction `visualize`

Dessine les boîtes OBB (polygones orientés) sur l'image et sauvegarde la figure.

In [56]:
# Palette de couleurs par classe (extension automatique si > 10 classes)
PALETTE = [
    "#e6194b", "#3cb44b", "#4363d8", "#f58231",
    "#911eb4", "#42d4f4", "#f032e6", "#bfef45",
    "#fabed4", "#469990"
]

def visualize(results, img_pil: Image.Image, conf: float, save_path: Path) -> None:
    fig, ax = plt.subplots(1, 1, figsize=(14, 18))
    ax.imshow(np.array(img_pil))
    ax.axis("off")
    ax.set_title(f"YOLO-gen OBB — seuil de confiance : {conf:.2f}", fontsize=14)

    legend_handles = {}
    has_detections = False

    for result in results:
        if result.obb is None or len(result.obb) == 0:
            continue

        has_detections = True
        names = result.names

        for box in result.obb:
            cls_id    = int(box.cls[0])
            score     = float(box.conf[0])
            cls_name  = names[cls_id]
            color     = PALETTE[cls_id % len(PALETTE)]

            # Coordonnées des 4 coins → tableau (4, 2)
            pts = box.xyxyxyxy[0].cpu().numpy().reshape(4, 2)

            # Polygone rempli (semi-transparent)
            poly_fill = Polygon(
                pts, closed=True,
                linewidth=0, edgecolor=color,
                facecolor=color, alpha=0.15
            )
            # Polygone contour
            poly_edge = Polygon(
                pts, closed=True,
                linewidth=1.5, edgecolor=color,
                facecolor="none", alpha=1.0
            )
            ax.add_patch(poly_fill)
            ax.add_patch(poly_edge)

            # Étiquette au coin supérieur gauche
            top_idx = pts[:, 1].argmin()
            tx, ty  = pts[top_idx]
            ax.text(
                tx, ty - 4,
                f"{cls_name} {score:.2f}",
                fontsize=7, color="white",
                bbox=dict(boxstyle="round,pad=0.2", facecolor=color, alpha=0.8, linewidth=0)
            )

            if cls_name not in legend_handles:
                legend_handles[cls_name] = Patch(color=color, label=cls_name)

    if not has_detections:
        h, w = np.array(img_pil).shape[:2]
        ax.text(
            w / 2, h / 2,
            "Aucune détection",
            fontsize=20, color="red",
            ha="center", va="center",
            bbox=dict(facecolor="white", alpha=0.7)
        )

    if legend_handles:
        ax.legend(
            handles=list(legend_handles.values()),
            loc="upper right",
            fontsize=9,
            framealpha=0.8
        )

    fig.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Figure sauvegardée : {save_path}")

## 6. Fonction `report_and_export`

Affiche un rapport console groupé par classe et exporte les détections en JSON.

In [57]:
def report_and_export(results, conf: float, save_path: Path) -> None:
    stats   = defaultdict(list)   # {classe: [scores]}
    regions = []

    for result in results:
        if result.obb is None or len(result.obb) == 0:
            continue

        names = result.names

        for box in result.obb:
            cls_id   = int(box.cls[0])
            score    = float(box.conf[0])
            cls_name = names[cls_id]
            pts      = box.xyxyxyxy[0].cpu().numpy().reshape(4, 2).tolist()

            stats[cls_name].append(score)
            regions.append({
                "class":      cls_name,
                "confidence": round(score, 4),
                "polygon":    [[round(x, 1), round(y, 1)] for x, y in pts]
            })

    # Rapport console
    print(f"\n{'Classe':<20} {'Effectif':>8} {'Conf. moy.':>10} {'Conf. max':>10}")
    print("-" * 52)
    for cls_name, scores in sorted(stats.items()):
        print(
            f"{cls_name:<20} {len(scores):>8} "
            f"{np.mean(scores):>10.3f} {max(scores):>10.3f}"
        )
    print(f"\nTotal : {len(regions)} détection(s)")

    # Export JSON
    payload = {
        "model":           HF_MODEL_ID,
        "paper":           "arXiv:2506.20326",
        "conf_threshold":  conf,
        "regions":         regions
    }
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    print(f"JSON exporté       : {save_path}")

## 7. Pipeline principal `main`

In [ ]:
def main():
    OUTPUT_DIR.mkdir(exist_ok=True)

    model         = load_model()
    results, conf = diagnose_thresholds(model, IMAGE_PATH)
    img_pil       = Image.open(IMAGE_PATH).convert("RGB")

    visualize(
        results, img_pil, conf,
        OUTPUT_DIR / "yologen_detections_croped.jpg"
    )
    report_and_export(
        results, conf,
        OUTPUT_DIR / "yologen_detections_croped.json"
    )
    bootstrap_detections(results, n_iterations=10000, ci=95)

    return model, results  # exposé pour les cellules suivantes

model_loaded, results_main = main()

## 8. Comparaison sur 4 tailles d'image

In [ ]:
def compare_imgsz(model: YOLO, image_path: str, sizes: list[int], conf: float = 0.05, iou: float = 0.45):
    """Lance l'inférence sur plusieurs tailles et affiche un tableau comparatif + grille de visualisation."""

    print(f"{'imgsz':>8} | {'Détections':>10} | {'Conf. moy.':>10} | Classes détectées")
    print("-" * 65)

    fig, axes = plt.subplots(1, len(sizes), figsize=(6 * len(sizes), 10))
    img_pil = Image.open(image_path).convert("RGB")

    for ax, imgsz in zip(axes, sizes):
        results = model.predict(
            source=image_path,
            conf=conf,
            iou=iou,
            imgsz=imgsz,
            verbose=False
        )

        detections = []
        class_counts = defaultdict(int)

        for r in results:
            if r.obb is None:
                continue
            for box in r.obb:
                cls_name = model.names[int(box.cls[0])]
                score    = float(box.conf[0])
                pts      = box.xyxyxyxy[0].cpu().numpy().reshape(4, 2)
                detections.append((cls_name, score, pts))
                class_counts[cls_name] += 1

        n        = len(detections)
        mean_conf = np.mean([d[1] for d in detections]) if detections else 0.0
        classes  = ", ".join(f"{k}({v})" for k, v in sorted(class_counts.items()))

        print(f"{imgsz:>8} | {n:>10} | {mean_conf:>10.3f} | {classes}")

        # Visualisation
        ax.imshow(np.array(img_pil))
        ax.axis("off")
        ax.set_title(f"imgsz={imgsz}\n{n} détection(s)", fontsize=11)

        for cls_name, score, pts in detections:
            cls_id = [k for k, v in model.names.items() if v == cls_name][0]
            color  = PALETTE[cls_id % len(PALETTE)]
            ax.add_patch(Polygon(pts, closed=True, linewidth=1.5, edgecolor=color, facecolor=color, alpha=0.15))
            ax.add_patch(Polygon(pts, closed=True, linewidth=1.5, edgecolor=color, facecolor="none"))
            top_idx = pts[:, 1].argmin()
            ax.text(pts[top_idx][0], pts[top_idx][1] - 4,
                    f"{cls_name} {score:.2f}", fontsize=6, color="white",
                    bbox=dict(facecolor=color, alpha=0.8, linewidth=0, boxstyle="round,pad=0.2"))

    plt.suptitle(f"Comparaison imgsz — conf={conf}, iou={iou}", fontsize=13, y=1.01)
    plt.tight_layout()
    out_path = OUTPUT_DIR / "compare_imgsz.jpg"
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"\nFigure sauvegardée : {out_path}")


# Lancement sur 4 tailles
model_loaded = load_model()
OUTPUT_DIR.mkdir(exist_ok=True)
compare_imgsz(model_loaded, IMAGE_PATH, sizes=[640, 1080, 1280, 1920], conf=0.05)

## 9. Bootstrap statistique sur les détections

In [ ]:
def bootstrap_detections(results, n_iterations: int = 10000, ci: float = 95.0):
    """
    Bootstrap sur les détections YOLO :
    - Rééchantillonne les détections avec remise n_iterations fois
    - Calcule l'intervalle de confiance sur la confiance moyenne par classe
    """
    # Extraire toutes les détections sous forme de liste {classe, score}
    detections = []
    for result in results:
        if result.obb is None:
            continue
        for box in result.obb:
            detections.append({
                "class": result.names[int(box.cls[0])],
                "score": float(box.conf[0])
            })

    if not detections:
        print("Aucune détection à bootstrapper.")
        return

    n = len(detections)
    classes = sorted(set(d["class"] for d in detections))
    alpha = (100 - ci) / 2

    print(f"Bootstrap : {n_iterations} itérations sur {n} détections\n")
    print(f"{'Classe':<25} {'Moy. obs.':>10} {'IC bas':>10} {'IC haut':>10} {'n':>5}")
    print("-" * 65)

    fig, axes = plt.subplots(1, len(classes), figsize=(4 * len(classes), 4), squeeze=False)

    for ax, cls_name in zip(axes[0], classes):
        cls_scores = np.array([d["score"] for d in detections if d["class"] == cls_name])
        observed   = np.mean(cls_scores)

        # Rééchantillonnage avec remise
        boot_means = np.array([
            np.mean(np.random.choice(cls_scores, size=len(cls_scores), replace=True))
            for _ in range(n_iterations)
        ])

        lo = np.percentile(boot_means, alpha)
        hi = np.percentile(boot_means, 100 - alpha)

        print(f"{cls_name:<25} {observed:>10.4f} {lo:>10.4f} {hi:>10.4f} {len(cls_scores):>5}")

        # Histogramme de la distribution bootstrap
        ax.hist(boot_means, bins=60, color="#4363d8", alpha=0.7, edgecolor="none")
        ax.axvline(observed, color="red",    linestyle="--", linewidth=1.5, label=f"obs={observed:.3f}")
        ax.axvline(lo,       color="orange", linestyle=":",  linewidth=1.2, label=f"IC {ci:.0f}%")
        ax.axvline(hi,       color="orange", linestyle=":",  linewidth=1.2)
        ax.set_title(cls_name, fontsize=9)
        ax.set_xlabel("Conf. moy.", fontsize=8)
        ax.legend(fontsize=7)

    plt.suptitle(f"Distributions bootstrap des confiances ({n_iterations} itérations)", fontsize=12)
    plt.tight_layout()
    out_path = OUTPUT_DIR / "bootstrap.jpg"
    fig.savefig(out_path, dpi=120, bbox_inches="tight")
    plt.show()
    print(f"\nFigure sauvegardée : {out_path}")


# Lancement — utilise les résultats du pipeline principal
results_main, conf_main = diagnose_thresholds(model_loaded, IMAGE_PATH)
bootstrap_detections(results_main, n_iterations=10000, ci=95)